## 1. Setup

Installs, imports, load .env

In [6]:
%pip install -q -r ../requirements.txt

import json
import os
import sys
from pathlib import Path

sys.path.append("../src")

import pandas as pd

print("Setup complete ✓")

Note: you may need to restart the kernel to use updated packages.
Setup complete ✓



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Config

Model names, API base URLs, temperature, max_tokens, fixed label list

In [7]:
from config import (
    DATA_PATH, RESULTS_DIR, LABELS, MODELS, API_PRICING_PER_1M,
    RUN_DATE, TEMPERATURE, MAX_TOKENS, OPENAI_API_KEY,
)

print("LABELS:", LABELS)
print("MODELS:", MODELS)
print("TEMPERATURE:", TEMPERATURE, " MAX_TOKENS:", MAX_TOKENS)
print("RUN_DATE:", RUN_DATE)
print("OPENAI_API_KEY loaded:", "✓" if OPENAI_API_KEY else "✗ — add it to .env")

LABELS: ['C', 'C#', 'C++', 'Go', 'Java', 'JavaScript', 'PHP', 'Python', 'Rust', 'TypeScript']
MODELS: {'top': {'provider': 'openai', 'model': 'gpt-5.6-sol', 'base_url': 'https://api.openai.com/v1'}, 'cheap': {'provider': 'openai', 'model': 'gpt-5.6-luna', 'base_url': 'https://api.openai.com/v1'}, 'local': {'provider': 'vllm', 'model': '<placeholder — teammate fills this in>', 'base_url': 'http://localhost:8000/v1'}}
TEMPERATURE: 0  MAX_TOKENS: 16
RUN_DATE: 2026-09-09
OPENAI_API_KEY loaded: ✗ — add it to .env


## 3. Prompt template

In [8]:
from prompt import build_prompt

# sanity check: see the exact frozen prompt every model receives
print(build_prompt('print("hello world")'))

You are a programming language classifier.

Identify the programming language used in the code snippet.

Choose exactly one language from this list:

C
C#
C++
Go
Java
JavaScript
PHP
Python
Rust
TypeScript

Return ONLY the language name, spelled exactly as it appears in the list above.
Do not provide an explanation, punctuation, or any other text.

Code:
print("hello world")



## 4. Load data

In [9]:
from collections import Counter
from run import load_items

ITEMS = load_items()

# Validate — fail loudly; never silently skip bad labels
invalid = [item["id"] for item in ITEMS if item["expected"] not in LABELS]
if invalid:
    raise ValueError(f"Items with unknown labels (fix before running): ids={invalid}")

print(f"Total items: {len(ITEMS)}")
print("\nLabel distribution:")
for lang, count in sorted(Counter(item["expected"] for item in ITEMS).items()):
    print(f"  {lang:<12} {count}")

Total items: 50

Label distribution:
  C            5
  C#           5
  C++          5
  Go           5
  Java         5
  JavaScript   5
  PHP          5
  Python       5
  Rust         5
  TypeScript   5


## 5. Run: Top API model

In [10]:
from run import run_model, write_csv

PER_ITEM_CSV = RESULTS_DIR / "per_item.csv"

# Calls the real Groq API for all 50 items and appends to results/per_item.csv.
# Re-running this cell re-calls the API and appends duplicate rows — delete
# per_item.csv first if you want a clean re-run.
top_rows = run_model("top", ITEMS)
write_csv(top_rows, path=PER_ITEM_CSV, append=True)

OSError: OPENAI_API_KEY not set — add it to .env before running

## 6. Run: Cheap API model

In [ ]:
cheap_rows = run_model("cheap", ITEMS)
write_csv(cheap_rows, path=PER_ITEM_CSV, append=True)

## 7. Run: Open-weights model

In [ ]:
# Local model has already been run separately and its 50 results
# are already stored in results/per_item.csv.
# Do NOT run it again here, otherwise we will duplicate the 50 local rows.

print("Local model: 50 existing benchmark rows loaded from per_item.csv")

## 8. Score

In [ ]:
# run.py already scores each row at call time using score.py's exact-match
# logic, so this just loads the accumulated results for analysis.
per_item = pd.read_csv(PER_ITEM_CSV)
print(f"Loaded {len(per_item)} rows across models: {sorted(per_item['model_key'].unique())}")
per_item.head()

## 9. Cost

In [ ]:
from cost import ApiPricing, api_cost_per_1k_requests, cost_at_volume

cost_rows = []
for model_key in ["top", "cheap"]:
    sub = per_item[per_item["model_key"] == model_key]
    if sub.empty:
        continue
    avg_in = sub["input_tokens"].mean()
    avg_out = sub["output_tokens"].mean()

    pricing = ApiPricing(
        price_per_1k_input=API_PRICING_PER_1M[model_key]["input"] / 1000,
        price_per_1k_output=API_PRICING_PER_1M[model_key]["output"] / 1000,
    )
    cost_per_1k = api_cost_per_1k_requests(avg_in, avg_out, pricing)

    cost_rows.append({
        "model_key": model_key,
        "avg_input_tokens": round(avg_in, 1),
        "avg_output_tokens": round(avg_out, 1),
        "cost_per_1k_requests_usd": round(cost_per_1k, 4),
        "cost_at_100x_traffic_usd": round(cost_at_volume(cost_per_1k / 1000, 100_000), 2),
    })

# Local model cost uses hardware $/hour instead of token pricing — add once
# hardware_note.md has real numbers, e.g.:
# from cost import local_cost_per_1k_requests
# local_cost_1k = local_cost_per_1k_requests(hardware_cost_per_hour=..., requests_per_hour=...)

cost_df = pd.DataFrame(cost_rows)
cost_df

## 10. Results summary table

In [ ]:
summary_rows = []
for model_key, group in per_item.groupby("model_key"):
    n_correct = group["correct"].sum()
    n_total = len(group)
    summary_rows.append({
        "model_key": model_key,
        "accuracy": f"{n_correct}/{n_total}",
        "accuracy_pct": round(100 * n_correct / n_total, 1),
        "p50_latency_ms": round(group["latency_ms"].quantile(0.50), 1),
        "p95_latency_ms": round(group["latency_ms"].quantile(0.95), 1),
        "parse_errors": (group["status"] == "parse_error").sum(),
        "timeouts": (group["status"] == "timeout").sum(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary_df

## 11. Sample wrong answers

In [ ]:
for model_key, group in per_item.groupby("model_key"):
    wrong = group[group["correct"] == False].head(3)
    print(f"\n=== {model_key}: wrong/error answers ===")
    for _, row in wrong.iterrows():
        print(f"  item {row['item_id']}: expected={row['expected']!r}  got={row['parsed']!r}  raw={row['raw_output']!r}  status={row['status']}")